In [7]:
# pip install git+https://github.com/huggingface/diffusers

import torch
from diffusers import ZImagePipeline

pipe = ZImagePipeline.from_pretrained(
    "./Z-Image",  # load from your local download
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=False,
)
pipe.to("cuda")

image = pipe(prompt="Powerful and seductive portrait of a Capricorn, a hot and sexy girl with a breast size of 85, a curvy girl with a big ass. Her skin color is fair and her hair is long and black and curly. She has big brown eyes. She has sexy plump lips. I want her naked and only wearing a thin sexy red lace thong. She is not wearing a bra and her nipples are red with but with a red cross-shaped tape on her nipples. Her nail's polish is red. I want this girl to be on top of a mountain and combine her with a goat (i.e., she has horns and a goat's tail and her horns and tail are sexy.).",
height=1024,
    width=1024,
    num_inference_steps=50,       # Z-Image-Turbo is distilled for very few steps
    guidance_scale=5.0).images[0]          # Turbo variant expects guidance_scale=0.0).images[0]
image.save("Pizza_fake.png")
pipe.to("cpu"); del pipe; torch.cuda.empty_cache()

100%|██████████| 50/50 [01:57<00:00,  2.35s/it]


In [4]:
import torch
from diffusers import ZImagePipeline
pipe = ZImagePipeline.from_pretrained(
    "./Z-Image",              # Foundation model — NOT -Turbo — required for CFG/negative prompts
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=False,
)
pipe.to("cuda")

prompt = (
    "A close-up richly detailed oil-painted fantasy character portrait of a dwarf lord, "
    "in the style of classic epic fantasy book cover illustration, painted with visible "
    "brushwork, deep glazes, and masterful realistic rendering; dark warm background fading "
    "from soft tan light into deep umber shadow; stocky and powerfully built with a broad, "
    "muscular upper body and short bowed legs; mismatched eyes — one deep obsidian black, "
    "the other a vivid piercing green — painted with subtle wet reflective highlights; his "
    "nose is reduced to a rough stub, most of it sliced away in battle, a deep jagged scar "
    "running from cheekbone to jaw, rendered with realistic skin texture and healed tissue "
    "detail; his jaw is uneven and asymmetrical, short bristly mismatched-colored hair falls "
    "unevenly, catching warm rim light against the dark background; he looks directly at the "
    "viewer with a shrewd, knowing half-smile, head tilted slightly, radiating cunning and "
    "quiet authority; he wears fine dark burnished plate armor with intricate gold filigree "
    "engraving across the chestplate and pauldrons, a heavy crimson cape fastened by a gold "
    "lion-head clasp draped over one shoulder, the fabric rendered with deep realistic folds, "
    "soft velvet sheen, and directional highlights; one hand rests near the pommel of a sword "
    "at his hip, fine detail in the rings and knuckles; dramatic Rembrandt-style side lighting "
    "falls across his face and armor from camera-left, carving strong contrast between warm "
    "lit highlights and deep soft shadow; softly blurred moody background with subtle vignette, "
    "in the tradition of classic fantasy portrait painting, realistic skin texture with fine "
    "pores and natural asymmetry, muted rich color palette of crimson, aged gold, and dark "
    "steel, elegant and painterly, gallery-quality digital oil painting, sharp focus on the "
    "face, noble and commanding presence despite his stature"
)

negative_prompt = (
    "flat cel-shaded animation, cartoon, anime style, vector art, chibi proportions, "
    "smooth plastic skin, glossy toon shading, thick black outlines, intact undamaged nose, "
    "symmetrical unscarred face, matching eye colors, blank vacant stare, looking away from "
    "camera, plain or drab clothing, timid expression, low detail, blurry, extra limbs, "
    "deformed hands, malformed face, text, watermark, signature, oversaturated flat colors, "
    "harsh overexposure, lens flare artifacts, unrealistic proportions, exaggerated scars "
    "beyond described, bright flat even lighting, washed-out colors, child-like features, "
    "tall average human proportions, non-dwarf body type, 3d render, video game render, "
    "airbrushed digital art"
)
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=1024,
    width=1024,
    num_inference_steps=50,        # 28–50 is the recommended range; 40 balances quality/speed
    guidance_scale=5,            # 3.0–5.0 recommended; lower = more natural, higher = more literal/prompt-adherent
    cfg_normalization=True,        # True = suppresses oversaturation/burn from CFG — better for photorealism
    cfg_truncation=0.85,           # disables CFG in the last ~15% of steps, avoids over-guided/plasticky fine detail
    max_sequence_length=1024,      # your prompt is long; default 512 tokens may truncate it
    generator=torch.Generator("cuda").manual_seed(42),  # reproducible results while you tune
    num_images_per_prompt=1,
).images[0]

image.save("Tyrion Lannister_by_AI.png")

pipe.to("cpu")
del pipe
torch.cuda.empty_cache()

100%|██████████| 50/50 [02:01<00:00,  2.42s/it]


In [ ]:
pipe.to("cpu"); del pipe; torch.cuda.empty_cache()

In [ ]:
from transformers.modeling_utils import PreTrainedModel

In [5]:
MODEL_NAME="/media/avidmech/data/Z_image_model_image_generation/Z-Image"          # the folder where you already downloaded the weights
INSTANCE_DIR="/media/avidmech/data/Z_image_model_image_generation/My dataset"
OUTPUT_DIR="/media/avidmech/data/Z_image_model_image_generation/trained-z-image-lora_for_brad_pitt_chracter"



In [1]:
cd diffusers/examples/dreambooth

/media/avidmech/data/Z_image_model_image_generation/diffusers/examples/dreambooth


/home/avidmech/miniconda3/envs/wan_i2v/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import torch

In [3]:
!accelerate launch train_dreambooth_lora_z_image.py \
  --pretrained_model_name_or_path="/media/avidmech/data/Z_image_model_image_generation/Z-Image" \
  --dataset_name="/media/avidmech/data/Z_image_model_image_generation/My dataset" \
  --caption_column="text" \
  --instance_prompt="a professional portrait photo" \
  --output_dir="/media/avidmech/data/Z_image_model_image_generation/trained-z-image-lora_for_brad_pitt_chracter" \
  --mixed_precision="bf16" \
  --gradient_checkpointing \
  --cache_latents \
  --resolution=720 \
  --train_batch_size=2 \
  --guidance_scale=5.0 \
  --use_8bit_adam \
  --gradient_accumulation_steps=4 \
  --optimizer="adamW" \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=100 \
  --rank=32 \
  --lora_alpha=32 \
  --num_train_epochs=20 \
  --validation_prompt="a professional studio portrait, cinematic lighting, photorealistic" \
  --validation_epochs=1 \
  --seed=0

07/14/2026 16:20:44 - INFO - __main__ - [RANK 0] Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: bf16

{'base_image_seq_len', 'max_image_seq_len', 'time_shift_type', 'use_exponential_sigmas', 'invert_sigmas', 'base_shift', 'stochastic_sampling', 'shift_terminal', 'use_beta_sigmas', 'use_karras_sigmas', 'max_shift'} was not found in config. Values will be initialized to default values.
All model checkpoint weights were used when initializing AutoencoderKL.

All the weights of AutoencoderKL were initialized from the model checkpoint at /media/avidmech/data/Z_image_model_image_generation/Z-Image.
If your task is similar to the task the model of the checkpoint was trained on, you can already use AutoencoderKL for predictions without further training.
Instantiating ZImageTransformer2DModel model under default dtype torch.bfloat16.
Loading checkpoint shards: 100%|████████████████| 2/2 [00:00<00:00, 106.15it/s]
All model

In [6]:
pwd

'/media/avidmech/data/Z_image_model_image_generation/diffusers/examples/dreambooth'

In [17]:
pipe.to("cpu"); del pipe; torch.cuda.empty_cache()

In [11]:
cd /media/avidmech/data/Z_image_model_image_generation

/media/avidmech/data/Z_image_model_image_generation


In [16]:
import torch
from diffusers import ZImagePipeline

pipe = ZImagePipeline.from_pretrained("/media/avidmech/data/Z_image_model_image_generation/Z-Image", torch_dtype=torch.bfloat16)
pipe.to("cuda")

pipe.load_lora_weights("/media/avidmech/data/Z_image_model_image_generation/trained-z-image-lora_for_brad_pitt_chracter")

image = pipe(
    prompt="write asexy picture of bradpit with grirl macking out in an steamy environment . be crfull on the body and mak edesirable for both man and woman looking at the picture it must be erotics",
    height=1024,
    width=1024,
    num_inference_steps=50,
    guidance_scale=5.0,
    generator=torch.Generator("cuda").manual_seed(42),
).images[0]

image.save("generted_new_lora.png")

100%|██████████| 50/50 [02:03<00:00,  2.47s/it]
